# Olist: Customer Churn & Lifetime Value
### What actually drives repeat purchase in a marketplace where 97% of customers buy only once

---

## Executive summary  *(the 60-second read)*

**Business question:** Olist wants to grow customer loyalty. Who churns, who is worth keeping, and where should retention spend go?

**What the data forced us to confront first:** of approx. 96,000 customers, only **2.2%** ever buy a second time, and just **1.3%** come back within a quarter. On a market this one-and-done, the textbook churn model ("flag anyone inactive for N days") is a trap — it scores 97% "accurate" by predicting everyone leaves, and teaches nothing. So the first deliverable here isn't a model; it's a **defensible re-definition of the problem**, built from evidence.

**Four findings a loyalty VP can act on:**

1. **Repeat purchase is only weakly predictable from a customer's first order** (cross-validated ROC-AUC approx.  **0.56**). Translation: on Olist you *cannot target your way to loyalty* from first-order behaviour. That is a strategic finding, not a modelling failure.
2. **What you sell beats how you serve.** Repeat rate swings approx. **4×** by category — furniture & fashion (approx. 2.3%) vs electronics, food, "cool stuff" (approx. 0.6%). Loyalty here is largely a **catalogue-mix outcome**, not a marketing outcome.
3. **The one operational lever that matters is not failing to deliver.** An order that never arrives collapses repeat rate to **0.16%** vs **1.32%** for delivered orders. Delivery *speed* barely moves the needle; delivery *failure* is fatal.
4. **Value is heavily concentrated.** Among the thin repeat layer, the **top 10% of customers hold approx. 66%** of the cohort's predicted 12-month value. Retention budget should follow value, not headcount.

**So what:** loyalty strategy on Olist should be **category-aware and acquisition-led** — win the second order through catalogue and delivery reliability, then concentrate retention economics on the small, high-value repeat tier that CLV identifies.

---


### How this notebook is built

It runs top-to-bottom on Kaggle with the **Brazilian E-Commerce Public Dataset by Olist** attached as input. The narrative is deliberate: we *disprove* the naive churn framing before adopting a defensible one, then model, explain, and value. Every code cell is commented so each decision is auditable.

**Reproducibility:** the loader finds the data wherever Kaggle mounts it, so this runs unchanged for anyone who copies it.

> **Note:** enable **Internet** in the notebook settings before running — the CLV section installs the `lifetimes` library.

In [ ]:
# ============================================================
# Imports, data load, and the notebook's visual "house style".
# Everything below inherits the styling defined here.
# ============================================================
import pandas as pd, numpy as np, glob, os
import matplotlib.pyplot as plt
from IPython.display import HTML, display

# --- Path-proof load: search for the files wherever Kaggle mounts them, so the
#     notebook never breaks on a re-mount and runs for anyone who copies it. ---
hit = glob.glob("/kaggle/input/**/olist_orders_dataset.csv", recursive=True)
assert hit, "Dataset not attached — add 'brazilian-ecommerce' by olistbr on the right panel."
PATH = os.path.dirname(hit[0]) + "/"

orders    = pd.read_csv(PATH + "olist_orders_dataset.csv", parse_dates=["order_purchase_timestamp"])
customers = pd.read_csv(PATH + "olist_customers_dataset.csv")

# --- THE KEY JOIN: customer_id is regenerated per order; customer_unique_id is the
#     actual person. Everything downstream depends on using the right key. ---
df = orders.merge(customers[["customer_id", "customer_unique_id"]], on="customer_id", how="left")
n_cust = df["customer_unique_id"].nunique()
print(f"{len(orders):,} orders | {n_cust:,} real customers")

# ---------------- HOUSE STYLE (McKinsey-lite) ----------------
NAVY, ACCENT, MUTE, LIGHT = "#0C2345", "#C8A24B", "#6B7280", "#D6DCE5"  # resume navy + gold

plt.rcParams.update({
    "figure.figsize": (9, 4.8), "figure.dpi": 110,
    "font.family": "DejaVu Sans", "font.size": 11,
    "axes.spines.top": False, "axes.spines.right": False, "axes.spines.left": False,
    "axes.edgecolor": MUTE, "axes.grid": True, "axes.axisbelow": True,
    "grid.color": LIGHT, "grid.linewidth": 0.8,
    "xtick.bottom": True, "ytick.left": False,   # drop y-ticks, keep faint grid
    "axes.prop_cycle": plt.cycler(color=[NAVY, ACCENT, MUTE, "#8FA3BF"]),
})

def banner(title, subtitle=""):
    """Navy section/chart banner. Sits ABOVE a chart or opens a section."""
    sub = f'<div style="color:#c9d2e0;font-size:12px;margin-top:2px;">{subtitle}</div>' if subtitle else ''
    display(HTML(f"""<div style="background:{NAVY};padding:12px 18px;border-radius:8px;
        border-left:5px solid {ACCENT};margin:14px 0 6px;">
        <div style="color:#fff;font-size:16px;font-weight:700;font-family:Georgia,serif;">{title}</div>
        {sub}</div>"""))

def mck(ax, title, subtitle="", source="Source: Olist Brazilian E-Commerce (2016–2018)"):
    """McKinsey-style action title INSIDE a chart: the takeaway is the headline,
    the chart is just the proof. Muted subtitle + source note beneath."""
    ax.set_title("")
    ax.text(0, 1.14, title, transform=ax.transAxes, fontsize=13.5, fontweight="bold", color=NAVY, ha="left")
    if subtitle: ax.text(0, 1.05, subtitle, transform=ax.transAxes, fontsize=10, color=MUTE, ha="left")
    if source:   ax.text(0, -0.22, source, transform=ax.transAxes, fontsize=8, color=MUTE, ha="left")

print("House style loaded.")


## 1. The join trap and the base-rate problem

`customer_id` is regenerated for every order, so joining on it makes every customer look unique: you would "prove" that *nobody* ever returns. Joining on `customer_unique_id` reveals the truth: approx. 3% repeat. That means naive "churn = won't return" has a approx. 97% base rate, so a model that blindly predicts *everyone churns* scores 97% accuracy and learns nothing. **Accuracy is a banned metric from here on.**

In [ ]:
# Repeat counts under the CORRECT key vs the trap key.
real_repeaters = (df.groupby("customer_unique_id")["order_id"].nunique() > 1).sum()
trap_repeaters = (orders.groupby("customer_id")["order_id"].nunique() > 1).sum()

print(f"Repeat rate (correct key) : {real_repeaters/n_cust:.2%}  ({real_repeaters:,} customers)")
print(f"Repeaters (trap key)      : {trap_repeaters}   <-- wrong join erases every repeat customer")


## 2. How long is "gone"? Let the returners set the line

We won't pick a churn horizon by gut. We measure how long customers who *did* return took to place their second order, then set the line where returns thin out. (Note the spike at day 0: we interrogate it in Section 3.)

In [ ]:
# Order each customer's purchases in time, then measure the gap from 1st to 2nd order.
df = df.sort_values(["customer_unique_id", "order_purchase_timestamp"])
df["seq"] = df.groupby("customer_unique_id").cumcount()          # 0 = first order, 1 = second, ...

first  = df[df.seq == 0].set_index("customer_unique_id")["order_purchase_timestamp"]
second = df[df.seq == 1].set_index("customer_unique_id")["order_purchase_timestamp"]
gap_days = ((second - first).dt.total_seconds() / 86400).dropna()

for h in [30, 60, 90, 120, 180]:
    print(f"  % of 2nd orders within {h:>3}d : {(gap_days <= h).mean():.1%}")

# --- CHART: banner on top, action-title inside ---
banner("How long repeat buyers take to come back",
       "Distribution of days between first and second order")
fig, ax = plt.subplots()
ax.hist(gap_days[gap_days <= 365], bins=50, color=NAVY, edgecolor="white", linewidth=0.4)
ax.axvline(90, color=ACCENT, ls="--", lw=2)
ax.text(96, ax.get_ylim()[1]*0.82, "90-day churn line\n(1 quarter)",
        color=ACCENT, fontsize=9, fontweight="bold")
ax.set_xlabel("Days from first to second purchase"); ax.set_yticks([])
mck(ax, "Repeat buyers return fast — and rarely after one quarter",
    "Median 28 days between first and second order; ~69% of all returns land inside 90 days.")
plt.show()


**Decision — churn horizon = 90 days.** It captures the bulk of genuine returns, sits on the elbow of the curve, and equals one business quarter — the cadence loyalty teams plan on. Customers who return later aren't a labelling failure; they're a separate *reactivation* segment.

## 3. Not every "repeat" is a repeat

That day-0 spike is suspicious. A time-gap check shows a large group whose "second order" lands the *same minute* as the first: a median gap of 0.0 minutes. That isn't loyalty; it's a single checkout that Olist split across multiple sellers into separate `order_id`s.

In [ ]:
# Zoom into the instant "repeats": how many are literally the same checkout?
gap_hrs = (second - first).dt.total_seconds() / 3600
same_minute = (gap_hrs <= 1).sum()
print(f"'Repeaters' with 2nd order within 1 hour : {same_minute:,}")
print(f"Median gap of that group (minutes)       : {(gap_hrs[gap_hrs <= 1] * 60).median():.1f}")
print("-> 0.0 min = same checkout split across sellers. NOT a real repeat purchase.")


### The fix: count purchase *events*, not `order_id`s

We collapse same-day fragments into one purchase event per customer. This removes approx. 850 phantom repeaters and leaves an honest cohort of approx. 2,150 — smaller, but real. This is the only population the CLV models are allowed to see.

In [ ]:
# A "purchase event" = one customer on one calendar day (same-day fragments merged).
df["order_day"] = df["order_purchase_timestamp"].dt.normalize()
events = df.groupby("customer_unique_id")["order_day"].nunique()

clean_repeaters = (events > 1).sum()
print(f"Clean repeat cohort       : {clean_repeaters:,}  ({clean_repeaters/n_cust:.2%})")
print(f"Phantom repeaters removed : {real_repeaters - clean_repeaters:,}")


## 4. Giving customers a fair chance (censoring)

A customer who first bought in the final 90 days of the data had no runway to return, yet a naive label would call them "churned." We drop them from the *labelable* base rather than mislabel them. The cost is small (approx. 9.5%), so a 90-day horizon is cheap.

In [ ]:
# Anyone whose first purchase is inside the last 90 days can't be fairly labelled — drop them.
END, HORIZON = orders["order_purchase_timestamp"].max(), 90
first_purchase = df.groupby("customer_unique_id")["order_purchase_timestamp"].min()
censored = (first_purchase > (END - pd.Timedelta(days=HORIZON))).sum()

print(f"Labelable customers       : {n_cust - censored:,}")
print(f"Censored (no fair chance) : {censored:,}  ({censored/n_cust:.1%})")


### Locked definition
- **Purchase event**: distinct order-day per `customer_unique_id` (same-day fragments collapsed)
- **Churn** — no second purchase event within **90 days** of the first
- **Labelable base** — approx. 86,900 customers (first purchase ≥ 90 days before 2018-10-17)
- **CLV cohort** — approx. 2,150 true repeaters; BG/NBD + Gamma-Gamma applied to these only


## 5. Can we predict who comes back?

We build a modelling table under one strict rule that mirrors real-world scoring: **every feature is knowable from the customer's FIRST order alone.** Predicting "will they return within 90 days" using anything from the second order would be leakage: the model would cheat, score beautifully, and collapse in production. Delivery time and review score of the *first* order are fair game: "did a slow delivery or a bad experience kill the repeat?" is precisely the loyalty question.

In [ ]:
# ============================================================
# Build the repeat-propensity modelling table.
# GOLDEN RULE: features come ONLY from each customer's first order.
# ============================================================
# Reload the tables we need (kept self-contained for clarity).
orders = pd.read_csv(PATH+"olist_orders_dataset.csv",
                     parse_dates=["order_purchase_timestamp",
                                  "order_delivered_customer_date",
                                  "order_estimated_delivery_date"])
customers = pd.read_csv(PATH+"olist_customers_dataset.csv")
items    = pd.read_csv(PATH+"olist_order_items_dataset.csv")
pays     = pd.read_csv(PATH+"olist_order_payments_dataset.csv")
reviews  = pd.read_csv(PATH+"olist_order_reviews_dataset.csv")
products = pd.read_csv(PATH+"olist_products_dataset.csv")
trans    = pd.read_csv(PATH+"product_category_name_translation.csv")

o = orders.merge(customers[["customer_id","customer_unique_id","customer_state"]],
                 on="customer_id", how="left")
o["order_day"] = o["order_purchase_timestamp"].dt.normalize()
END = o["order_purchase_timestamp"].max()

# ---- TARGET: did a 2nd purchase EVENT happen within 90 days of the first? ----
firsts = o.groupby("customer_unique_id")["order_purchase_timestamp"].min().rename("first_ts").reset_index()
firsts["first_day"] = firsts["first_ts"].dt.normalize()
o2 = o.merge(firsts[["customer_unique_id","first_day"]], on="customer_unique_id")
o2["days_since_first"] = (o2["order_day"] - o2["first_day"]).dt.days
# >0 excludes same-day fragments; <=90 is the churn window
o2["is_repeat"] = (o2["days_since_first"] > 0) & (o2["days_since_first"] <= 90)
y = o2.groupby("customer_unique_id")["is_repeat"].max().astype(int).rename("y_repeat")

# labelable base: first purchase needs a full 90-day runway before the data ends
labelable = firsts.loc[firsts["first_ts"] <= END - pd.Timedelta(days=90), "customer_unique_id"]

# ---- FEATURES: strictly from the FIRST order ----
first_order = (o.sort_values("order_purchase_timestamp")
                 .groupby("customer_unique_id").first().reset_index()
                 [["customer_unique_id","order_id","customer_state","order_purchase_timestamp",
                   "order_delivered_customer_date","order_estimated_delivery_date"]])

# basket economics of the first order
it = items.groupby("order_id").agg(price=("price","sum"), freight=("freight_value","sum"),
        n_items=("order_item_id","count"), n_sellers=("seller_id","nunique"),
        product_id=("product_id","first")).reset_index()
it = it.merge(products[["product_id","product_category_name"]], on="product_id", how="left") \
       .merge(trans, on="product_category_name", how="left")

pay = pays.groupby("order_id").agg(payment_value=("payment_value","sum"),
        installments=("payment_installments","max"),
        payment_type=("payment_type","first")).reset_index()

rev = reviews.groupby("order_id")["review_score"].mean().rename("review_score").reset_index()

feat = (first_order.merge(it, on="order_id", how="left")
                   .merge(pay, on="order_id", how="left")
                   .merge(rev, on="order_id", how="left"))
feat["delivery_days"] = (feat["order_delivered_customer_date"] - feat["order_purchase_timestamp"]).dt.days
feat["late_delivery"] = (feat["order_delivered_customer_date"] > feat["order_estimated_delivery_date"]).astype("Int64")
feat["freight_ratio"] = feat["freight"] / feat["price"].replace(0, pd.NA)
feat["purchase_month"] = feat["order_purchase_timestamp"].dt.month
feat["purchase_dow"]   = feat["order_purchase_timestamp"].dt.dayofweek

# attach target + restrict to the labelable base
model_df = feat.merge(y, on="customer_unique_id")
model_df = model_df[model_df["customer_unique_id"].isin(labelable)].copy()

print(f"Modelling rows      : {len(model_df):,}")
print(f"Positive rate (y=1) : {model_df['y_repeat'].mean():.2%}   (~76:1 imbalance)")


**A note on honest evaluation.** An early attempt that aggressively class-weighted the rare positives (`scale_pos_weight approx.  76`) actually scored *worse* on ranking — heavy weighting chases a few positives and overfits noise. The version below drops the weighting, regularises, and reports **cross-validated out-of-fold** predictions — no peeking, the honest ceiling. We judge on **PR-AUC and lift**, never accuracy.

In [ ]:
# ---- Prepare the model matrix (X, y) ----
from lightgbm import LGBMClassifier

d = model_df.copy()
# FIX: a never-delivered order was wrongly flagged "on time". Undelivered is its own
# (strong) signal, so we flag it explicitly and blank the misleading late flag.
d["is_delivered"] = d["delivery_days"].notna().astype(int)
d.loc[d["delivery_days"].isna(), "late_delivery"] = np.nan

cat_feats = ["product_category_name_english", "payment_type", "customer_state"]
num_feats = ["price","freight","n_items","n_sellers","payment_value","installments",
             "review_score","delivery_days","late_delivery","freight_ratio",
             "purchase_month","purchase_dow","is_delivered"]
for c in cat_feats: d[c] = d[c].astype("category")
X, y = d[cat_feats + num_feats], d["y_repeat"]
print("Model matrix:", X.shape)


In [ ]:
# ---- Honest cross-validated evaluation ----
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import average_precision_score, roc_auc_score

oof = np.zeros(len(X))
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for tr, va in skf.split(X, y):
    m = LGBMClassifier(n_estimators=400, learning_rate=0.03, num_leaves=15,
                       min_child_samples=300, subsample=0.8, colsample_bytree=0.8,
                       reg_lambda=1.0, random_state=42, n_jobs=-1, verbose=-1)
    m.fit(X.iloc[tr], y.iloc[tr])
    oof[va] = m.predict_proba(X.iloc[va])[:, 1]   # score each fold with a model that never saw it

base = y.mean()
print(f"ROC-AUC (OOF) : {roc_auc_score(y, oof):.4f}   (0.50 = coin flip)")
print(f"PR-AUC  (OOF) : {average_precision_score(y, oof):.4f}   "
      f"(base {base:.4f}, lift {average_precision_score(y, oof)/base:.1f}x)")

# Deciles by predicted propensity (ranked, to avoid tied-probability bucketing issues).
dec = pd.DataFrame({"y": y.values, "p": oof})
dec["decile"] = pd.qcut(dec["p"].rank(method="first"), 10, labels=[f"D{i}" for i in range(1, 11)])


In [ ]:
# ---- CHART: how sharply do the deciles separate? ----
rate = dec.groupby("decile", observed=True)["y"].mean() * 100
top_lift = rate.iloc[-1] / (base * 100)

banner("The model sorts customers only weakly — and that is the finding",
       "Actual repeat rate by predicted-propensity decile")
fig, ax = plt.subplots(figsize=(8.5, 4.2))
ax.bar(range(10), rate.values, color=[MUTE]*9 + [NAVY], width=0.72)
ax.axhline(base*100, color=ACCENT, ls="--", lw=1.4)
ax.text(-0.4, base*100, f"base {base*100:.2f}%", color=ACCENT, fontsize=8.5,
        fontweight="bold", va="bottom", ha="left")
ax.set_xticks(range(10)); ax.set_xticklabels(rate.index, fontsize=9)
ax.set_yticks([]); ax.set_xlabel("Predicted-propensity decile (D10 = highest)")
mck(ax, f"Even the top decile repeats at only ~{top_lift:.1f}x the base rate",
    "First-order signals barely predict return — you cannot target your way to loyalty here.")
plt.show()


## 6. What *does* drive repeat? Raw conditional rates

When a model is deliberately weak, a weak model's feature importances are not trustworthy: so we read the drivers straight from the data as conditional repeat rates. These are undeniable and need no model.

In [ ]:
# Read drivers directly as conditional repeat rates vs the base rate.
d = model_df.copy()
d["is_delivered"] = d["delivery_days"].notna().astype(int)
base = d["y_repeat"].mean()

def rate_by(col, min_n=200):
    g = d.groupby(col, observed=True)["y_repeat"].agg(n="count", rep="mean")
    g = g[g["n"] >= min_n].copy()
    g["lift"] = (g["rep"] / base).round(2)
    g["repeat_%"] = (g["rep"] * 100).round(2)
    return g[["n","repeat_%","lift"]].sort_values("repeat_%", ascending=False)

print(f"BASE repeat rate: {base*100:.2f}%\n")
print("— Did the order even arrive? —");            print(rate_by("is_delivered", min_n=1), "\n")
d["deliv_bucket"] = pd.cut(d["delivery_days"], [-1,3,7,15,999], labels=["0–3d","4–7d","8–15d","16d+"])
print("— Delivery speed —");                        print(rate_by("deliv_bucket"), "\n")
print("— First-order review score —");              print(rate_by("review_score"), "\n")


In [ ]:
# ---- CHART: the category story (the sharpest slide) ----
cat = (d.groupby("product_category_name_english", observed=True)["y_repeat"]
         .agg(n="count", rep="mean"))
cat = cat[cat["n"] >= 300].copy()
top5, bot5 = cat.nlargest(5, "rep"), cat.nsmallest(5, "rep")
plot_df = pd.concat([bot5, top5]).sort_values("rep")

banner("What you sell predicts loyalty more than how you serve",
       "Repeat rate by category vs base — bottom 5 and top 5 (min. 300 buyers)")
fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(range(len(plot_df)), plot_df["rep"].values*100, color=[MUTE]*5 + [NAVY]*5, height=0.7)
ax.axvline(base*100, color=ACCENT, ls="--", lw=1.4)
ax.text(base*100, -1.2, f"base {base*100:.2f}%", color=ACCENT, fontsize=8.5, fontweight="bold", ha="center")
ax.set_yticks(range(len(plot_df)))
ax.set_yticklabels([c.replace("_"," ") for c in plot_df.index], fontsize=9)
ax.set_xlabel("Repeat rate (%)"); ax.set_xticks(np.arange(0, 3, 0.5))
mck(ax, "Top categories repeat at ~4x the rate of the bottom",
    "Furniture & fashion drive return; electronics, food & 'cool stuff' are one-and-done.")
plt.tight_layout(); plt.show()


## 7. Lifetime value of the repeat cohort

CLV is estimated with **BG/NBD** (future transaction counts) and **Gamma-Gamma** (monetary value): applied *only* to the approx. 2,150 true repeaters, because both models require `frequency > 0`. Forcing them onto 96k one-time buyers would degenerate to naive average-order-value, defeating the purpose.

*Caveat:* with a small, low-frequency cohort the Gamma-Gamma shape parameter comes in below 1, which makes the *mean* CLV unstable and long-tailed. We therefore headline the **concentration** of value (robust) rather than the absolute mean.

In [ ]:
# ============================================================
# CLV — BG/NBD + Gamma-Gamma on the TRUE repeat cohort only.
# ============================================================
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lifetimes"], check=False)
from lifetimes import BetaGeoFitter, GammaGammaFitter
from lifetimes.utils import summary_data_from_transaction_data

# Rebuild a clean transaction log: one row per real purchase EVENT, with money.
orders = pd.read_csv(PATH+"olist_orders_dataset.csv", parse_dates=["order_purchase_timestamp"])
cust   = pd.read_csv(PATH+"olist_customers_dataset.csv")
items  = pd.read_csv(PATH+"olist_order_items_dataset.csv")

o = orders.merge(cust[["customer_id","customer_unique_id"]], on="customer_id", how="left")
o["order_day"] = o["order_purchase_timestamp"].dt.normalize()
val = items.groupby("order_id").agg(order_val=("price","sum")).reset_index()
val["order_val"] += items.groupby("order_id")["freight_value"].sum().values
o = o.merge(val, on="order_id", how="left")

# collapse same-day fragments into one event, summing value (phantom fix applied to money)
tx = (o.groupby(["customer_unique_id","order_day"], observed=True)
        .agg(order_val=("order_val","sum")).reset_index())
tx = tx[tx["order_val"] > 0]

obs_end = o["order_day"].max()
rfm = summary_data_from_transaction_data(
        tx, "customer_unique_id", "order_day",
        monetary_value_col="order_val", observation_period_end=obs_end)

rep = rfm[rfm["frequency"] > 0].copy()   # repeat cohort only
print(f"Repeat cohort for CLV : {len(rep):,}   (full base {len(rfm):,})")

bgf = BetaGeoFitter(penalizer_coef=0.01);  bgf.fit(rep["frequency"], rep["recency"], rep["T"])
gg  = GammaGammaFitter(penalizer_coef=0.01); gg.fit(rep["frequency"], rep["monetary_value"])

rep["clv_12m"] = gg.customer_lifetime_value(
        bgf, rep["frequency"], rep["recency"], rep["T"], rep["monetary_value"],
        time=12, freq="D", discount_rate=0.01)   # ~12.7% annual
rep["p_alive"]        = bgf.conditional_probability_alive(rep["frequency"], rep["recency"], rep["T"])
rep["pred_purch_6m"]  = bgf.conditional_expected_number_of_purchases_up_to_time(
        180, rep["frequency"], rep["recency"], rep["T"])

top10 = rep.nlargest(int(len(rep)*0.1), "clv_12m")["clv_12m"].sum() / rep["clv_12m"].sum()
print(f"Median 12m CLV : R$ {rep['clv_12m'].median():,.2f}")
print(f"Top 10% share of cohort value : {top10:.1%}")


In [ ]:
# ---- CHART: value concentration ----
banner("A thin repeat layer carries outsized value",
       "12-month predicted CLV across the repeat cohort")
clv = rep["clv_12m"].clip(upper=rep["clv_12m"].quantile(0.99))   # trim tail for readability
fig, ax = plt.subplots(figsize=(9, 4.4))
ax.hist(clv, bins=40, color=NAVY, edgecolor="white", linewidth=0.4)
med = rep["clv_12m"].median()
ax.axvline(med, color=ACCENT, ls="--", lw=1.6)
ax.text(med, ax.get_ylim()[1]*0.9, f"  median R$ {med:,.0f}", color=ACCENT, fontsize=9, fontweight="bold")
ax.set_xlabel("Predicted 12-month CLV (R$)"); ax.set_yticks([])
mck(ax, f"The top 10% of repeaters hold {top10:.0%} of cohort value",
    "Value is concentrated — retention spend should follow CLV, not headcount.")
plt.tight_layout(); plt.show()

# Export the scored cohort of repeat customers.
out = rep.reset_index()[["customer_unique_id","frequency","recency","T",
        "monetary_value","p_alive","pred_purch_6m","clv_12m"]]
out.to_csv("olist_clv_repeat_cohort.csv", index=False)
print(f"Exported {len(out):,} scored customers -> olist_clv_repeat_cohort.csv")


## 8. Strategic implications

1. **Don't buy a churn-scoring tool for this problem.** First-order behaviour barely predicts return (ROC-AUC approx.  0.56); spend on prediction here has low ceiling.
2. **Treat loyalty as a catalogue decision.** Repeat rate is approx. 4× higher in furniture/fashion than in electronics/food. Category mix, bundling, and cross-sell into sticky categories will move retention more than CRM tactics.
3. **Make delivery reliability non-negotiable.** Failed delivery is the clearest killer of repeat (0.16% vs 1.32%). Speed is secondary; *arrival* is everything.
4. **Concentrate retention economics.** approx. 66% of cohort value sits in the top 10% of repeaters: identify them via CLV and protect them; treat the rest as acquisition-and-convert.

**Bottom line:** on Olist, the lever isn't predicting who leaves — it's engineering the *second purchase* through catalogue and delivery, then defending the small, high-value tier that CLV reveals.

---
*Author: Sagar Sharma · Customer & loyalty analytics.*
